# Final Group Tasks

For every task, aim for a small but complete scientific workflow: clear array/table meaning, explicit assumptions, numerical checks, at least one diagnostic plot, and a short written interpretation. Do not try to build a large project. A clear, correct, well-explained result is better than a long unfinished notebook.


## Suggested group workflow

1. Spend the first 5 minutes reading the task and dividing roles.
2. Build the minimum working version first.
3. Add checks before improving plots or performance.
4. Keep a short written decision log: what you computed, what you checked, and what you concluded.
5. Prepare a 7-minute summary for the group discussion.

A good final answer should contain:

- one or two clearly labelled figures;
- one small table or diagnostic dictionary;
- explicit validation checks;
- a short interpretation in plain scientific language.


In [ ]:
# Common imports for the final group session.
# Individual tasks may add a few extra imports when needed.

from pathlib import Path
import json
import timeit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---

# Task 1 — Coordinate-aware field diagnostics

You are given a two-dimensional scalar field on a physical grid. The goal is to treat the field as a numerical experiment object: keep coordinates and values together, compute diagnostics before plotting, and make a plot that preserves physical coordinates.

Use the setup below.


In [ ]:
rng = np.random.default_rng(2026)

x = np.linspace(-6.0, 6.0, 240)
y = np.linspace(-4.0, 4.0, 180)
X, Y = np.meshgrid(x, y, indexing="ij")

x0, y0 = 1.2, -0.8
sigma_x, sigma_y = 1.0, 1.6

rho = np.exp(-(
    (X - x0)**2 / (2.0 * sigma_x**2)
    + (Y - y0)**2 / (2.0 * sigma_y**2)
))

# A weak deterministic modulation makes the field asymmetric but still non-negative.
rho *= 1.0 + 0.12 * np.cos(2.5 * X + 0.8 * Y)
rho = np.maximum(rho, 0.0)

# A plotting trap: this outlier should not silently control the whole colour scale.
rho_with_outlier = rho.copy()
rho_with_outlier[18, 140] = 10.0 * rho.max()

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("rho.shape:", rho.shape)

Your tasks are:

1. State the meaning of both axes of `rho`. Which axis corresponds to `x`, and which axis corresponds to `y`?
2. Create a dictionary called `field_packet` containing at least `x`, `y`, `X`, `Y`, `rho`, `dx`, `dy`, and a short description.
3. Add numerical contracts checking that:
   - `x` and `y` are one-dimensional and strictly increasing;
   - `X`, `Y`, and `rho` have compatible shapes;
   - `rho` is finite and non-negative.
4. Compute a diagnostic dictionary containing:
   - integral of `rho` over the domain;
   - centre of mass;
   - width in the `x` and `y` directions;
   - maximum location.
5. Make a coordinate-aware plot of `rho` using either `pcolormesh` or `imshow` with physical axes. Overlay the centre of mass and maximum location.
6. Make a second plot of `rho_with_outlier`, but choose the colour limits deliberately, for example using the 1st and 99th percentiles.
7. Write three or four sentences explaining which diagnostics would reveal a wrong axis convention, a wrong colour scale, or a wrong grid.

**Optional extension:** Repeat the diagnostics for a small sweep of `sigma_x` values, store the results in a `DataFrame`, and plot the measured `x`-width as a function of `sigma_x`.


In [ ]:
#####################
# Your solution here
#####################

---

# Task 2 — A small honest machine-learning workflow

You are given synthetic data from a scientific regression problem. The goal is not to use a sophisticated model, but to build an honest numerical workflow: split the data, fit preprocessing on the training subset only, compare with a baseline, train a simple model, and inspect residuals.

Use the setup below.


In [ ]:
rng = np.random.default_rng(314)

N = 750

time = rng.uniform(0.0, 12.0, size=N)
field = rng.uniform(-1.5, 1.5, size=N)
temperature = 300.0 + 6.0 * rng.normal(size=N)

noise = 0.12 * rng.normal(size=N)

y = (
    0.80 * np.sin(1.2 * time)
    + 0.35 * field**2
    - 0.04 * (temperature - 300.0)
    + noise
)

raw_features = pd.DataFrame({
    "time": time,
    "field": field,
    "temperature_K": temperature,
})

raw_features.head()

Your tasks are:

1. Build the feature matrix `X` from the columns of `raw_features`. State the meaning of axis 0 and axis 1 of `X`.
2. Split the data into training, validation, and test subsets using a reproducible random permutation. Suggested fractions: 70%, 15%, 15%.
3. Standardize the features using only the training subset. Apply the same mean and scale to validation and test data.
4. Compute a baseline model that always predicts the mean of `y_train`. Report its training and validation loss.
5. Fit a linear model on the standardized features. You may use `np.linalg.lstsq`; a gradient-descent implementation is welcome but not required.
6. Report training, validation, and test loss for the fitted model. Check whether the model improves over the baseline.
7. Make one plot comparing predicted and true values on the test set, and one residual diagnostic plot.
8. Write a short interpretation: Did preprocessing avoid leakage? Is the model better than the baseline? Do the residuals show systematic structure?

**Optional extension:** Implement full-batch gradient descent for the same model and compare its final validation loss with the least-squares solution.


In [ ]:
#####################
# Your solution here
#####################

---

# Task 3 — A reproducible labelled data pipeline

You are given two raw CSV files:

```text
events.csv       event-level measurements
runs.csv         run-level metadata
```

The goal is to build a compact data pipeline with portable paths, typed tables, a validated join, visible cleaning rules, and saved outputs.

Run the setup cell below to create the raw files.


In [ ]:
rng = np.random.default_rng(1729)

project_root = Path("final_group_pipeline")
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

run_ids = [f"run_{i:03d}" for i in range(1, 8)]
detectors = ["A", "B", "C"]

runs = pd.DataFrame({
    "run_id": run_ids,
    "field_T": np.linspace(0.05, 0.40, len(run_ids)),
    "temperature_K": 300.0 + rng.normal(scale=0.5, size=len(run_ids)),
    "operator": rng.choice(["shift_1", "shift_2"], size=len(run_ids)),
})

n_events = 1200

events = pd.DataFrame({
    "event_id": np.arange(n_events),
    "run_id": rng.choice(run_ids + ["run_missing"], size=n_events, p=[0.14]*7 + [0.02]),
    "detector": rng.choice(detectors, size=n_events),
    "energy_GeV": rng.gamma(shape=4.0, scale=0.8, size=n_events),
    "chi2": rng.gamma(shape=1.4, scale=0.8, size=n_events),
    "signal": rng.normal(loc=12.0, scale=2.0, size=n_events),
    "exposure_s": rng.uniform(0.3, 2.0, size=n_events),
})

# Inject a few problematic rows that should be handled explicitly.
events.loc[5, "energy_GeV"] = -1.0
events.loc[12, "signal"] = np.nan
events.loc[25, "exposure_s"] = 0.0
events.loc[37, "chi2"] = np.inf

events.to_csv(raw_dir / "events.csv", index=False)
runs.to_csv(raw_dir / "runs.csv", index=False)

print("Created:", raw_dir / "events.csv")
print("Created:", raw_dir / "runs.csv")

Your tasks are:

1. Define a `config` dictionary containing at least `min_energy_GeV`, `max_chi2`, and a short text description of the analysis.
2. Write `load_inputs(project_root)` that reads both CSV files using `pathlib` paths. Choose useful dtypes for `run_id` and `detector`.
3. Write `clean_events(events, runs, config)` that:
   - joins events with run metadata using `validate="many_to_one"` and `indicator=True`;
   - removes rows with missing metadata;
   - removes non-finite or physically invalid numerical rows;
   - applies the energy and chi-square cuts;
   - adds `signal_rate = signal / exposure_s`.
4. Write `summarize(clean)` that groups by `run_id` and `detector`, and returns at least `n_events`, `mean_energy_GeV`, `mean_rate`, `std_rate`, and `field_T`.
5. Save the summary table and the configuration under `final_group_pipeline/results/tables`.
6. Make one diagnostic plot of mean signal rate versus magnetic field, with detector labels or a legend.
7. Write a short interpretation: which assumptions are visible in the pipeline, which rows were removed, and why is the validated join important?

**Optional extension:** Add an audit table with counts of rows removed because of missing metadata, invalid numerical values, and final selection cuts. Save it together with the summary.


In [ ]:
#####################
# Your solution here
#####################

---

# Task 4 — Measurement-guided optimization of a numerical kernel

You are given a clear but slow Python implementation of a finite-difference residual. The goal is to follow a disciplined optimization workflow: establish a baseline, validate a candidate implementation, benchmark representative inputs, and interpret the result.

Use the setup below.


In [ ]:
def make_residual_problem(n):
    x = np.linspace(-8.0, 8.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x) + 0.03 * np.sin(4.0 * x)
    return x, u, dx


def residual_baseline(u, dx):
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + u[i] * (u[i]**2 - 1.0)

    return r


def benchmark_call(call, *, number=1, repeat=5):
    samples = timeit.repeat(call, number=number, repeat=repeat)
    return np.array(samples) / number


sizes = np.array([1_000, 3_000, 10_000, 30_000])

Your tasks are:

1. Implement `residual_candidate(u, dx)` using vectorized NumPy operations. Keep the same boundary convention as the baseline.
2. For each value in `sizes`, validate that `residual_candidate` agrees with `residual_baseline` using an appropriate numerical tolerance.
3. Benchmark both implementations using repeated timings. Record at least the best and median time per call.
4. Store the benchmark results in a `DataFrame` with columns such as `n`, `implementation`, `best_s`, and `median_s`.
5. Make a log-log plot of runtime versus `n` for both implementations.
6. Write a compact decision report:
   - What was measured?
   - What was the representative input?
   - Did the candidate remain correct?
   - Did the optimization mainly change the scaling law or the constant factor?
   - Is the added complexity justified?

**Optional extension:** If Numba is available, add a compiled loop implementation as a third candidate. Separate compilation time from steady-state runtime.


In [ ]:
#####################
# Your solution here
#####################

---

# Task 5 — Parameter sweep as a compact scientific report

You are given a family of damped oscillatory signals. The goal is to represent the full parameter sweep as structured arrays, reduce it to labelled diagnostics, visualize the result, and write a short scientific interpretation.

Use the setup below.


In [ ]:
t = np.linspace(0.0, 14.0, 1200)

gamma_values = np.array([0.03, 0.06, 0.10, 0.16, 0.25])
omega_values = np.array([1.5, 2.0, 2.5, 3.0])

# Axes: damping gamma, angular frequency omega, time t
U = (
    np.exp(-gamma_values[:, None, None] * t[None, None, :])
    * np.sin(omega_values[None, :, None] * t[None, None, :])
)

print("U.shape:", U.shape)

Your tasks are:

1. State the meaning of each axis of `U`. Which axis should be reduced to compute one diagnostic per parameter pair?
2. Compute at least four diagnostics for every `(gamma, omega)` pair:
   - energy-like integral $\int U(t)^2\,dt$;
   - maximum absolute amplitude;
   - time at which the maximum absolute amplitude occurs;
   - final value at the last time point.
3. Build a labelled `DataFrame` with one row per parameter pair and columns for `gamma`, `omega`, and the diagnostics.
4. Make one curve plot showing all time traces for a selected value of `gamma` or a selected value of `omega`.
5. Make one heatmap-like plot of the energy-like diagnostic as a function of `gamma` and `omega`.
6. Save the diagnostic table to `final_group_sweep/results/tables/sweep_summary.csv`.
7. Write a short interpretation: Which parameter pair retains the largest energy-like value? How does damping change the diagnostics? Which plot was most useful?

**Optional extension:** Repeat the same diagnostic calculation in a streaming style, without storing the full three-dimensional array `U`. Compare the memory use of the stored array with the streaming approach.


In [ ]:
#####################
# Your solution here
#####################

---

# Final discussion checklist

Before presenting, check that your notebook section can be run from a clean kernel. Then prepare a short explanation addressing:

1. What numerical or data object did your task start from?
2. What assumptions did you make explicit?
3. What checks did you add before trusting the result?
4. What did your main diagnostic plot show?
5. What would be the next improvement if you had another 30 minutes?
